In [1]:
# =============================================================================
# CONFIGURATION - Adjust these before running
# =============================================================================

# Number of samples to process (set to None for all 101 samples)
NUM_SAMPLES: int | None = None  # Start with 5 for testing, then set to None

# Concurrency for async API calls
MAX_CONCURRENT_REQUESTS = 10

# Model to use
MODEL = "gpt-5.1"

In [2]:
import os
import json
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI
from datasets import load_dataset, Dataset
from tqdm.asyncio import tqdm_asyncio
import pandas as pd
from pathlib import Path

load_dotenv(Path(os.getcwd()).parents[1] / ".env")
client = AsyncOpenAI()  # Uses OPENAI_API_KEY env var

/Users/charliemasters/Desktop/synced_vm_002/arcane_extension/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = load_dataset("ScaleAI/researchrubrics")
df = pd.DataFrame(ds["train"])

# Apply sample limit if set
if NUM_SAMPLES is not None:
    df = df.head(NUM_SAMPLES)
    print(f"Processing first {NUM_SAMPLES} samples (set NUM_SAMPLES=None for all)")
else:
    print(f"Processing all {len(df)} samples")

Processing all 101 samples


In [4]:
ABLATION_SYSTEM_PROMPT = """You are an expert at modifying research requests to test whether AI agents ask clarifying questions THROUGHOUT a task (not just at the start).

CRITICAL CONSTRAINT: The ablated prompt must be SHORTER than the original. You are REMOVING context, not adding it. If the original is 400 chars, the ablated version should be ~200-350 chars.

Goal: Create a version where:
- An agent who asks questions MID-TASK (after discovering tradeoffs) can score ~100%
- An agent who doesn't ask, or only asks upfront questions, scores ~30-50%

## STEP 1: Distinguish Two Types of Ambiguity

UPFRONT-ASKABLE (we do NOT want this):
- "What format do you want?" - can ask before starting
- "How long should it be?" - can ask before starting  
- "Who is the audience?" - can ask before starting
These let an agent ask everything upfront, then execute without further interaction.

MID-TASK DISCOVERABLE (we WANT this):
- "I found 3 competing frameworks (X, Y, Z) - which matters most to you?" - only askable after research
- "Source A says X, Source B says Y - how should I weigh conflicting evidence?" - emerges from research
- "This topic has both theoretical foundations and practical applications - should I go deeper on theory or pivot to implementation?" - discovered mid-task
These REQUIRE the agent to do research first, then realize they need clarification.

## STEP 2: Analyze Implicit Criteria for Mid-Task Triggers

For each high-weight Implicit Criterion, ask: "When would an agent discover they need this?"
- If discoverable before research → keep in prompt (or it's just upfront ambiguity)
- If discoverable only AFTER finding options/tradeoffs → REMOVE from prompt

Examples:
- "Focus on practical over theoretical" → Remove. Agent discovers this matters when they find both perspectives exist.
- "Include case studies" → Remove only if cases vs theory is a tradeoff discovered mid-research.
- "Arrow's theorem constraints" → Remove. Agent discovers this when researching consensus mechanisms.

## STEP 3: Strategic Removal (SUBTRACT, don't add)

Remove phrases that reveal:
- COMPARATIVE preferences (which approach to prioritize)
- SYNTHESIS preferences (how to weigh conflicting sources)
- DEPTH decisions (how deep to go once subtopics are found)
- FRAMING choices (only clear once content structure emerges)

DO NOT add helpful structure, examples, or clarifying text. Only remove.

## BAD vs GOOD Examples

BAD ABLATION (added text, longer):
Original: "Create a plan for July 4th in DC with activities."
Ablated: "Create a plan for July 4th in DC. For each activity, explain when it happens and why it's interesting. Include any useful logistics."
Problem: Added helpful structure! Now agent doesn't need to ask anything.

GOOD ABLATION (removed preferences, shorter):
Original: "Create a detailed day-by-day itinerary for July 4th weekend in DC, focusing on family-friendly activities with indoor backup options for weather."
Ablated: "Create a plan for July 4th in DC."
Why it works: Agent must research, discover there are family vs adult options, indoor vs outdoor tradeoffs, then ask: "I found several fireworks spots - do you prefer crowds or quiet? Should I include indoor backups?"

Return JSON:
{
  "ablated_prompt": "SHORTER version with preferences removed",
  "removed_elements": ["what was removed → which mid-task question it forces"],
  "rubric_analysis": "Which implicit criteria require mid-task discovery? Estimated score loss without asking?",
  "ablation_reasoning": "What will the agent discover mid-research that forces them to ask?"
}"""

In [5]:
from pydantic import BaseModel


class AblationResult(BaseModel):
    rubric_analysis: str  # Which implicit criteria targeted, estimated % score lost
    ablation_reasoning: str  # How this creates mid-task ambiguity
    removed_elements: list[str]  # What was removed and which criterion it targets
    ablated_prompt: str

In [6]:
async def ablate_prompt(
    sample_id: str, original_prompt: str, rubrics: list[dict]
) -> tuple[str, dict]:
    """Ablate a single prompt using GPT-4o (async)."""

    # Format rubrics for context (especially Implicit Criteria)
    implicit_criteria = [r for r in rubrics if r.get("axis") == "Implicit Criteria"]
    explicit_criteria = [r for r in rubrics if r.get("axis") == "Explicit Criteria"]

    rubric_context = f"""
IMPLICIT CRITERIA (preferences the AI should ask about):
{json.dumps(implicit_criteria, indent=2)}

EXPLICIT CRITERIA (should remain in prompt):
{json.dumps(explicit_criteria, indent=2)}
"""

    try:
        response = await client.chat.completions.parse(
            model=MODEL,
            messages=[
                {"role": "system", "content": ABLATION_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": f"Original prompt:\n{original_prompt}\n\nRubric criteria:\n{rubric_context}",
                },
            ],
            response_format=AblationResult,
        )
        result = response.choices[0].message.parsed
        return sample_id, {
            "ablated_prompt": result.ablated_prompt,
            "removed_elements": result.removed_elements,
            "rubric_analysis": result.rubric_analysis,
            "ablation_reasoning": result.ablation_reasoning,
        }
    except Exception as e:
        print(f"Error on {sample_id}: {e}")
        return sample_id, None

In [7]:
async def process_all_ablations(df: pd.DataFrame) -> dict:
    """Process all ablations concurrently with rate limiting."""

    # Create semaphore to limit concurrent requests
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async def rate_limited_ablate(sample_id, prompt, rubrics):
        async with semaphore:
            return await ablate_prompt(sample_id, prompt, rubrics)

    # Create tasks for all rows
    tasks = [
        rate_limited_ablate(row["sample_id"], row["prompt"], row["rubrics"])
        for _, row in df.iterrows()
    ]

    # Run with progress bar
    results_list = await tqdm_asyncio.gather(*tasks, desc="Ablating prompts")

    # Convert to dict, filtering out failures
    results = {sample_id: result for sample_id, result in results_list if result is not None}

    return results


# Run the async processing
results = await process_all_ablations(df)
print(f"Completed {len(results)}/{len(df)} ablations")

# Show any failures
failed = len(df) - len(results)
if failed > 0:
    print(f"Failed: {failed} ablations")

Ablating prompts: 100%|██████████| 101/101 [03:53<00:00,  2.31s/it]

Completed 101/101 ablations


In [8]:
# Add ablation columns to dataframe
df["ablated_prompt"] = df["sample_id"].map(lambda x: results.get(x, {}).get("ablated_prompt", ""))
df["removed_elements"] = df["sample_id"].map(
    lambda x: results.get(x, {}).get("removed_elements", [])
)
df["rubric_analysis"] = df["sample_id"].map(lambda x: results.get(x, {}).get("rubric_analysis", ""))
df["ablation_reasoning"] = df["sample_id"].map(
    lambda x: results.get(x, {}).get("ablation_reasoning", "")
)

# Validate all rows have ablations
missing = df[df["ablated_prompt"] == ""]
print(f"Missing ablations: {len(missing)}")

Missing ablations: 0


In [9]:
# Show examples for manual review
for idx in range(len(df)):
    row = df.iloc[idx]
    print(f"\n{'=' * 80}")
    print(f"Sample {idx} - Domain: {row['domain']}")
    print(f"{'=' * 80}")

    print(f"\nORIGINAL ({len(row['prompt'])} chars):")
    print(row["prompt"])

    print(f"\nABLATED ({len(row['ablated_prompt'])} chars):")
    print(row["ablated_prompt"])

    print("\nRUBRIC ANALYSIS:")
    print(row["rubric_analysis"])

    print("\nABLATION REASONING:")
    print(row["ablation_reasoning"])

    print("\nREMOVED ELEMENTS:")
    for elem in row["removed_elements"]:
        print(f"  - {elem}")

    print(f"\nRUBRICS ({len(row['rubrics'])} criteria):")
    for rubric in row["rubrics"]:
        axis = rubric.get("axis", "Unknown")
        weight = rubric.get("weight", 0)
        criterion = rubric.get("criterion", "")
        print(f"  [{axis}] (w={weight}) {criterion}")


Sample 0 - Domain: Current Events

ORIGINAL (413 chars):
I want to create a plan for July 4, 2025, i.e., Independence Day in Washington DC. I would like an itinerary of all the things to do and all the activities that are planned for Independence Day. Create a plan for the whole day and also extend it to the weekend, if required. Provide some reviews or explain why one should visit the place or engage in the activity. Add any additional information that is required.

ABLATED (201 chars):
I want to create a plan for July 4, 2025 (Independence Day) in Washington, DC. Please give me an itinerary of things to do and activities that are planned for that day. Create a plan for the whole day.

RUBRIC ANALYSIS:
Key mid-task ambiguities now: (1) Balance between packed, all-day schedule vs a lighter plan—agent only sees this once they assemble many options; (2) How much of the weekend to use and whether to leave DC vs stay local; (3) Emphasis on formal scheduled events (parade, concert, firewor

In [10]:
# Compare length reduction
df["original_length"] = df["prompt"].str.len()
df["ablated_length"] = df["ablated_prompt"].str.len()
df["length_reduction"] = (
    (df["original_length"] - df["ablated_length"]) / df["original_length"] * 100
)

print("Length reduction statistics:")
print(df["length_reduction"].describe())

# Count removed elements
df["num_removed"] = df["removed_elements"].apply(len)
print(f"\nAverage elements removed: {df['num_removed'].mean():.1f}")

Length reduction statistics:
count    101.000000
mean      24.935101
std       30.055379
min      -93.137255
25%       11.891892
50%       28.115016
75%       46.403712
max       71.857923
Name: length_reduction, dtype: float64

Average elements removed: 5.9


In [11]:
# Prepare dataset for upload
# Keep original columns + add ablation columns
hf_dataset = Dataset.from_pandas(
    df[
        [
            "sample_id",
            "prompt",  # original
            "ablated_prompt",  # new
            "removed_elements",  # new
            "rubric_analysis",  # new - which criteria targeted, % score lost
            "ablation_reasoning",  # new - how mid-task ambiguity is created
            "domain",
            "conceptual_breadth",
            "logical_nesting",
            "exploration",
            "rubrics",
        ]
    ]
)

print(hf_dataset)

Dataset({
    features: ['sample_id', 'prompt', 'ablated_prompt', 'removed_elements', 'rubric_analysis', 'ablation_reasoning', 'domain', 'conceptual_breadth', 'logical_nesting', 'exploration', 'rubrics'],
    num_rows: 101
})


In [12]:
# Uses cached credentials from `huggingface-cli login`
from huggingface_hub import HfApi

# Get username
api = HfApi()
user_info = api.whoami()
username = user_info["name"]
print(f"Uploading as: {username}")

DATASET_NAME = f"{username}/researchrubrics-ablated"
# hf_dataset.push_to_hub(DATASET_NAME, private=False)
print(f"Dataset uploaded to: https://huggingface.co/datasets/{DATASET_NAME}")

Uploading as: cm2435cm2435cm2435
Dataset uploaded to: https://huggingface.co/datasets/cm2435cm2435cm2435/researchrubrics-ablated


In [13]:
# Clear results from memory if needed
# del results
print("Done! Dataset uploaded to HuggingFace.")

Done! Dataset uploaded to HuggingFace.
